In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        """
        Args:
            d_model: 输入(出)特征维度
            num_heads: 多头数量
            dropout: dropout比率
        """
        super().__init__()
        assert d_model%num_heads==0, "d_model必须能被num_heads整除"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model//num_heads

        # 四个线性层: QKV+输出投影
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)

        self.dropout = nn.Dropout(dropout)

    def forward(self, query, key, value, mask=None):
        """
        Args:
            query: (batch_size, seq_len_q, d_model)
            key: (batch_size, seq_len_k, d_model)
            value: (batch_size, seq_len_v, d_model)
            mask: (batch_size, seq_len_q, seq_len_k)
        Returns:
            output: (batch_size, seq_len_q, d_model)
            attention_weights: (batch_size, num_heads, seq_len_q, seq_len_k)
        """
        batch_size = query.size(0)

        # 1. 线性投影并分头
        # (batch_size, seq_len, d_model) -> (batch_size, seq_len, num_heads, d_k)
        Q = self.W_q(query).view(batch_size, -1, self.num_heads, self.d_k)
        K = self.W_k(query).view(batch_size, -1, self.num_heads, self.d_k)
        V = self.W_v(query).view(batch_size, -1, self.num_heads, self.d_k)

        # 2. 转置以便进行批量矩阵乘法
        # (batch_size, num_heads, seq_len, d_k)
        Q = Q.transpose(1,2)
        K = K.transpose(1,2)
        V = V.transpose(1,2)

        # 3. 计算缩放点积注意力
        scores = Q@K.transpose(-2, -1)/self.d_k**0.5